In [0]:
USE CATALOG league_records;

USE SCHEMA gold;

WITH final_snapshot AS (
    SELECT *
    FROM silver.intervals
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY match_id, participant_pos_id
        ORDER BY minute DESC
    ) = 1
),

match_stats_at_end AS (
    SELECT
        match_id,
        participant_pos_id,
        minute,
        level,
        kills,
        deaths,
        assists,
        (cs + jungle_cs) AS cs,
        total_gold
    FROM final_snapshot
),
-------------------------------------------------------------------------------------------
-- item_id_array -> flattened_item_ids -> named_item_build
-------------------------------------------------------------------------------------------
item_id_array AS (
    SELECT
        match_id,
        participant_pos_id,
        FILTER(
            ARRAY(item_0, item_1, item_2, item_3, item_4, item_5, item_6),
            x -> x IS NOT NULL
        ) AS item_ids
    FROM final_snapshot
),

flattened_item_ids AS (
    SELECT
        match_id,
        participant_pos_id,
        explode(item_ids) AS item_id
    FROM item_id_array
),

named_item_build AS (
    SELECT fi.match_id, fi.participant_pos_id,
        SORT_ARRAY(COLLECT_LIST(ir.item_name)) AS item_build
    FROM flattened_item_ids AS fi
    LEFT JOIN silver.items_ref AS ir
        ON ir.item_id = fi.item_id
        AND ir.__END_AT IS NULL
    GROUP BY fi.match_id, fi.participant_pos_id
)
-------------------------------------------------------------------------------------------
-- Join interval snapshot with matches, players, and resolved item-name build array
-------------------------------------------------------------------------------------------
SELECT
    se.match_id,
    se.participant_pos_id,
    mat.game_duration,
    ps.team,
    (ps.team = mat.winning_team) AS win,
    ps.champion_name,
    ps.champion_role,
    se.level,
    se.kills,
    se.deaths,
    se.assists,
    se.cs,
    se.total_gold,
    nb.item_build,
    GREATEST(mat.game_duration - se.minute * 60, 0) AS unlogged_duration
FROM match_stats_at_end AS se
JOIN silver.matches AS mat
    ON mat.match_id = se.match_id
JOIN silver.players AS ps
    ON ps.match_id = se.match_id
    AND ps.participant_pos_id = se.participant_pos_id
LEFT JOIN named_item_build AS nb
    ON nb.match_id = se.match_id
    AND nb.participant_pos_id = se.participant_pos_id
ORDER BY se.match_id, se.participant_pos_id
;